# **`CATE-HGN — Google Colab implementation`**

In [2]:
# CELL 1 — Install dependencies and check GPU
!pip -q install numpy pandas scikit-learn tqdm

import os, json, math, random, zipfile, glob, re, gc, subprocess, sys
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))


Device: cuda
Tesla T4


In [3]:
# CELL 2 — Download DBLP V11 archive to Colab
# The archive is large. Check available disk space before downloading.
DATA_DIR = Path("/content/cate_hgn_data")
RAW_DIR = DATA_DIR / "raw"
EXTRACT_DIR = RAW_DIR / "extracted"
PROC_DIR = DATA_DIR / "processed"
RESULT_DIR = Path("/content/cate_hgn_results")
for d in [RAW_DIR, EXTRACT_DIR, PROC_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DBLP_URL = "https://lfs.aminer.cn/misc/dblp.v11.zip"
ZIP_PATH = RAW_DIR / "dblp.v11.zip"

if not ZIP_PATH.exists():
    !wget -c -O /content/cate_hgn_data/raw/dblp.v11.zip https://lfs.aminer.cn/misc/dblp.v11.zip
else:
    print("Archive already exists:", ZIP_PATH, "size(GB)=", round(ZIP_PATH.stat().st_size/1e9, 2))

assert ZIP_PATH.exists() and ZIP_PATH.stat().st_size > 0, "Download failed or archive is empty."
print("Archive size (GB):", round(ZIP_PATH.stat().st_size/1e9, 2))


Archive already exists: /content/cate_hgn_data/raw/dblp.v11.zip size(GB)= 3.92
Archive size (GB): 3.92


In [4]:
# CELL 3 — Inspect archive and extract candidate data files
with zipfile.ZipFile(ZIP_PATH) as zf:
    members = [m for m in zf.infolist() if not m.is_dir()]
    print("Files in archive:", len(members))
    for m in members[:40]:
        print(m.filename, round(m.file_size/1e6, 2), "MB")
    bad_member = zf.testzip()
    if bad_member:
        raise RuntimeError("Corrupt archive member: " + bad_member)

# Extract only likely metadata files, avoiding unnecessary archive contents.
candidate_members = [
    m for m in members
    if m.filename.lower().endswith((".json", ".jsonl", ".txt", ".gz"))
]
print("Candidate metadata files:", [m.filename for m in candidate_members[:20]])

with zipfile.ZipFile(ZIP_PATH) as zf:
    for m in candidate_members:
        # Extract candidates to a folder, preserving basename.
        out = EXTRACT_DIR / Path(m.filename).name
        if not out.exists():
            with zf.open(m) as src, open(out, "wb") as dst:
                while True:
                    chunk = src.read(8 * 1024 * 1024)
                    if not chunk: break
                    dst.write(chunk)

found = []
for pattern in ("*.jsonl", "*.json", "*.txt", "*.gz"):
    found.extend(EXTRACT_DIR.rglob(pattern))
print("Extracted candidate paths:")
for p in found[:30]: print(p, round(p.stat().st_size/1e9, 3), "GB")

assert found, "No JSON/JSONL/TXT/GZ data member found. Inspect archive listing above."


Files in archive: 1
dblp_papers_v11.txt 12020.67 MB
Candidate metadata files: ['dblp_papers_v11.txt']
Extracted candidate paths:
/content/cate_hgn_data/raw/extracted/dblp_papers_v11.txt 12.021 GB


In [5]:
# CELL 4 — Ultra-low-RAM JSONL sampling

import json
import random
import gzip
from pathlib import Path

MAX_PAPERS = 500
MAX_TERMS = 1000
MAX_TERMS_PER_PAPER = 5

MIN_YEAR, MAX_YEAR = 1990, 2020
random.seed(42)

# Prefer the largest uncompressed .json/.jsonl/.txt candidate.
data_candidates = [p for p in found if p.suffix.lower() in (".jsonl", ".json", ".txt")]
DATA_FILE = max(data_candidates, key=lambda p: p.stat().st_size)

JSONL_PATH = Path(DATA_FILE)

def open_dataset(path):
    if str(path).endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8", errors="ignore")
    return open(path, "r", encoding="utf-8", errors="ignore")

def compact_authors(authors):
    result = []

    for author in authors or []:
        if isinstance(author, dict):
            name = author.get("name") or author.get("id")
        else:
            name = author

        if name:
            result.append(str(name)[:100])

    return result[:10]

rows = []
valid_count = 0

with open_dataset(JSONL_PATH) as f:
    for line_number, line in enumerate(f, 1):
        try:
            p = json.loads(line)
        except (json.JSONDecodeError, TypeError):
            continue

        try:
            year = int(p.get("year", 0))
        except (ValueError, TypeError):
            continue

        if not MIN_YEAR <= year <= MAX_YEAR:
            continue

        record = {
            "id": str(p.get("id", "")),
            "year": year,
            "n_citation": int(p.get("n_citation", 0) or 0),
            "authors": compact_authors(p.get("authors", [])),
            "venue": str(p.get("venue", ""))[:150],
            "title": str(p.get("title", ""))[:500],
            "abstract": str(p.get("abstract", ""))[:1000]
        }

        valid_count += 1

        if len(rows) < MAX_PAPERS:
            rows.append(record)
        else:
            j = random.randrange(valid_count)
            if j < MAX_PAPERS:
                rows[j] = record

        if line_number % 100000 == 0:
            print("Lines:", line_number, "| Stored:", len(rows))

print("Completed.")
print("Papers retained:", len(rows))
print("Example:", rows[0] if rows else "No records found")



Lines: 100000 | Stored: 500
Lines: 200000 | Stored: 500
Lines: 300000 | Stored: 500
Lines: 400000 | Stored: 500
Lines: 500000 | Stored: 500
Lines: 600000 | Stored: 500
Lines: 900000 | Stored: 500
Lines: 1000000 | Stored: 500
Lines: 1100000 | Stored: 500
Lines: 1200000 | Stored: 500
Lines: 1300000 | Stored: 500
Lines: 1400000 | Stored: 500
Lines: 1500000 | Stored: 500
Lines: 1600000 | Stored: 500
Lines: 1700000 | Stored: 500
Lines: 1800000 | Stored: 500
Lines: 1900000 | Stored: 500
Lines: 2000000 | Stored: 500
Lines: 2200000 | Stored: 500
Lines: 2300000 | Stored: 500
Lines: 2400000 | Stored: 500
Lines: 2500000 | Stored: 500
Lines: 2600000 | Stored: 500
Lines: 2700000 | Stored: 500
Lines: 2800000 | Stored: 500
Lines: 2900000 | Stored: 500
Lines: 3000000 | Stored: 500
Lines: 3100000 | Stored: 500
Lines: 3200000 | Stored: 500
Lines: 3300000 | Stored: 500
Lines: 3400000 | Stored: 500
Lines: 3500000 | Stored: 500
Lines: 3600000 | Stored: 500
Lines: 3700000 | Stored: 500
Lines: 3800000 | Stor

In [8]:
# CELL 5 — Construct paper/author/venue/term graph and labels
# Label used here is provisional: total citations / years since publication through 2020.
# Verify exact citation observation-window details against the paper before claiming
# exact numerical reproduction. Citation edges are NOT used for message passing.

import re

# Define helper functions that were missing from the environment
def clean_text(text):
    if not text:
        return ""
    # Lowercase, keep letters, numbers, hyphens, and single spaces
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def author_names(row):
    # Extract authors list
    return row.get("authors") or []

def venue_name(row):
    # Extract venue information
    venue = row.get("venue")
    if not venue:
        return ""
    # Clean dictionaries represented as strings
    if venue.startswith("{") and "'raw':" in venue:
        try:
            # Simple regex to pull out the raw venue name
            match = re.search(r"'raw'\s*:\s*'([^']+)'", venue)
            if match:
                return match.group(1)
            match = re.search(r'"raw"\s*:\s*"([^"]+)"', venue)
            if match:
                return match.group(1)
        except Exception:
            pass
    return clean_text(venue)

paper_ids = [str(r["id"]) for r in rows]
paper_index = {pid:i for i,pid in enumerate(paper_ids)}
n_papers = len(rows)
paper_year = np.array([int(r["year"]) for r in rows], dtype=np.int64)
citation_count = np.array([max(0.0, float(r.get("n_citation") or 0)) for r in rows], dtype=np.float32)
exposure_years = np.maximum(1, 2020 - paper_year + 1)
labels = citation_count / exposure_years

titles = [clean_text(r.get("title")) for r in rows]
abstracts = [clean_text(r.get("abstract") or r.get("indexed_abstract")) for r in rows]
documents = [(t + " " + a).strip() for t,a in zip(titles, abstracts)]

author_to_id, venue_to_id = {}, {}
pa_src, pa_dst = [], []
pv_src, pv_dst = [], []
for pi, r in enumerate(tqdm(rows, desc="Building author/venue links")):
    for name in set(author_names(r)):
        if name not in author_to_id: author_to_id[name] = len(author_to_id)
        pa_src.append(pi); pa_dst.append(author_to_id[name])
    venue = venue_name(r)
    if venue:
        if venue not in venue_to_id: venue_to_id[venue] = len(venue_to_id)
        pv_src.append(pi); pv_dst.append(venue_to_id[venue])

vectorizer = TfidfVectorizer(
    max_features=MAX_TERMS, stop_words="english",
    ngram_range=(1,2), min_df=2, max_df=0.90,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_-]{2,}\b"
)
tfidf = vectorizer.fit_transform(documents)
terms = vectorizer.get_feature_names_out().tolist()
pt_src, pt_dst, pt_weight = [], [], []
for pi in range(tfidf.shape[0]):
    start, end = tfidf.indptr[pi], tfidf.indptr[pi+1]
    idx = tfidf.indices[start:end]
    val = tfidf.data[start:end]
    if len(idx):
        top = np.argsort(val)[-MAX_TERMS_PER_PAPER:]
        for j in top:
            pt_src.append(pi); pt_dst.append(int(idx[j])); pt_weight.append(float(val[j]))

# Train: pre-2014; validation: 2014; test: 2015-2020.
train_idx = np.where(paper_year < 2014)[0]
val_idx = np.where(paper_year == 2014)[0]
test_idx = np.where((paper_year >= 2015) & (paper_year <= 2020))[0]
if min(len(train_idx), len(val_idx), len(test_idx)) == 0:
    raise RuntimeError(f"Empty temporal split: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}. Increase MAX_PAPERS or inspect years.")

graph = {
    "paper_author": torch.tensor([pa_src, pa_dst], dtype=torch.long),
    "paper_venue": torch.tensor([pv_src, pv_dst], dtype=torch.long),
    "paper_term": torch.tensor([pt_src, pt_dst], dtype=torch.long),
    "paper_term_weight": torch.tensor(pt_weight, dtype=torch.float32),
    "labels": torch.tensor(labels, dtype=torch.float32),
    "train_idx": torch.tensor(train_idx, dtype=torch.long),
    "val_idx": torch.tensor(val_idx, dtype=torch.long),
    "test_idx": torch.tensor(test_idx, dtype=torch.long),
    "meta": {
        "papers": n_papers, "authors": len(author_to_id), "venues": len(venue_to_id),
        "terms": len(terms), "paper_author_edges": len(pa_src),
        "paper_venue_edges": len(pv_src), "paper_term_edges": len(pt_src)
    }
}
print(json.dumps(graph["meta"], indent=2))
print("Split sizes:", len(train_idx), len(val_idx), len(test_idx))

Building author/venue links:   0%|          | 0/500 [00:00<?, ?it/s]

{
  "papers": 500,
  "authors": 1507,
  "venues": 407,
  "terms": 603,
  "paper_author_edges": 1516,
  "paper_venue_edges": 495,
  "paper_term_edges": 1830
}
Split sizes: 341 42 117


In [9]:
# CELL 6 — CATE-HGN-style PyTorch model
# Resource-bounded approximation:
# - one shared embedding space for papers/authors/venues/terms
# - relation-aware composition and per-relation neighbor attention
# - JSD-style cross-type mutual-information regularizer
# - DEC-style self-training clustering + cluster-conditioned masking
# It is not a bit-for-bit reproduction of all equations/three-way attention/TE-BERT loop.

class ClusterAware(nn.Module):
    def __init__(self, dim, n_clusters):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(n_clusters, dim) * 0.02)
        self.mask_logits = nn.Parameter(torch.zeros(n_clusters, dim))

    def forward(self, h):
        dist2 = torch.cdist(h, self.centers).pow(2)
        q = 1.0 / (1.0 + dist2)
        q = q / q.sum(dim=1, keepdim=True).clamp_min(1e-9)
        mask = torch.sigmoid(q @ self.mask_logits)
        return h * mask, q

    @staticmethod
    def target(q):
        freq = q.sum(0).clamp_min(1e-9)
        p = q.pow(2) / freq
        return (p / p.sum(1, keepdim=True).clamp_min(1e-9)).detach()

def cluster_kl(q):
    p = ClusterAware.target(q)
    return F.kl_div(q.clamp_min(1e-9).log(), p, reduction="batchmean")

class RelationAttention(nn.Module):
    def __init__(self, dim, n_rel=3):
        super().__init__()
        self.rel_emb = nn.Parameter(torch.randn(n_rel, dim) * 0.02)
        self.node_att = nn.ModuleList([nn.Linear(dim*3, 1) for _ in range(n_rel)])
        self.type_att = nn.Linear(dim*3, 1)
        self.msg_proj = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_rel)])
        self.out = nn.Linear(dim, dim)

    def forward(self, hp, context_embs, edge_indices, edge_weights):
        # edge_indices: [2,E], row 0=paper index, row 1=context-node index.
        relation_outputs, relation_logits = [], []
        n = hp.size(0)
        for rid, (ctx, edge, weights) in enumerate(zip(context_embs, edge_indices, edge_weights)):
            dst, src = edge[0], edge[1]
            if dst.numel() == 0:
                agg = torch.zeros_like(hp)
            else:
                neigh = ctx[src]
                center = hp[dst]
                rel = self.rel_emb[rid].expand_as(neigh)
                composed = neigh * rel
                score_in = torch.cat([center, composed, center * composed], dim=-1)
                score = self.node_att[rid](score_in).squeeze(-1)
                score = score + torch.log(weights.clamp_min(1e-8))
                # Stable softmax grouped by destination node.
                score_exp = torch.exp(score - score.max().detach())
                denom = torch.zeros(n, device=hp.device).index_add_(0, dst, score_exp)
                alpha = score_exp / denom[dst].clamp_min(1e-9)
                msg = self.msg_proj[rid](composed)
                agg = torch.zeros_like(hp).index_add_(0, dst, msg * alpha.unsqueeze(-1))
            relation_outputs.append(agg)
            relation_logits.append(self.type_att(torch.cat([hp, agg, hp*agg], dim=-1)))
        beta = torch.softmax(torch.cat(relation_logits, dim=1), dim=1)
        stacked = torch.stack(relation_outputs, dim=1)
        return F.relu(hp + self.out((stacked * beta.unsqueeze(-1)).sum(1)))

class CATEHGNColab(nn.Module):
    def __init__(self, n_p, n_a, n_v, n_t, dim=100, clusters=10, layers=2):
        super().__init__()
        self.p = nn.Embedding(n_p, dim)
        self.a = nn.Embedding(max(1,n_a), dim)
        self.v = nn.Embedding(max(1,n_v), dim)
        self.t = nn.Embedding(max(1,n_t), dim)
        self.gnn = nn.ModuleList([RelationAttention(dim) for _ in range(layers)])
        self.ca = nn.ModuleList([ClusterAware(dim, clusters) for _ in range(layers)])
        self.reg = nn.ModuleList([nn.Linear(dim, 1) for _ in range(layers)])
        self.disc = nn.ParameterList([nn.Parameter(torch.eye(dim)) for _ in range(layers-1)])
        for emb in [self.p, self.a, self.v, self.t]:
            nn.init.xavier_uniform_(emb.weight)

    def forward(self, g, return_aux=False):
        dev = self.p.weight.device
        hp = self.p.weight
        ca_emb = self.a.weight
        ve_emb = self.v.weight
        te_emb = self.t.weight

        edge_indices = [g["paper_author"].to(dev), g["paper_venue"].to(dev), g["paper_term"].to(dev)]
        edge_weights = [
            torch.ones(edge_indices[0].size(1), device=dev),
            torch.ones(edge_indices[1].size(1), device=dev),
            g["paper_term_weight"].to(dev)
        ]
        contexts = [ca_emb, ve_emb, te_emb]
        hidden_states, qs, predictions = [], [], []
        for layer, cluster, regressor in zip(self.gnn, self.ca, self.reg):
            hp = layer(hp, contexts, edge_indices, edge_weights)
            hp_masked, q = cluster(hp)
            hidden_states.append(hp_masked)
            qs.append(q)
            predictions.append(regressor(hp_masked).squeeze(-1))

        if not return_aux:
            return predictions[-1]

        mi_loss = torch.tensor(0.0, device=dev)
        if len(hidden_states) > 1:
            x, y = hidden_states[-2], hidden_states[-1]
            W = self.disc[0]
            positive = (x @ W * y).sum(-1)
            negative_y = y[torch.randperm(y.size(0), device=dev)]
            negative = (x @ W * negative_y).sum(-1)
            mi_loss = F.softplus(-positive).mean() + F.softplus(negative).mean()

        ca_loss = sum(cluster_kl(q) for q in qs)
        consistency = torch.tensor(0.0, device=dev)
        if len(qs) > 1:
            consistency = F.kl_div(qs[-1].clamp_min(1e-9).log(), qs[-2].detach(), reduction="batchmean")
        return predictions[-1], {"mi":mi_loss, "cluster":ca_loss, "consistency":consistency,
                                  "predictions":predictions, "assignments":qs}

model = CATEHGNColab(
    n_p=n_papers, n_a=len(author_to_id), n_v=len(venue_to_id), n_t=len(terms),
    dim=100, clusters=10, layers=2
).to(DEVICE)
print("Parameters:", sum(p.numel() for p in model.parameters()))


Parameters: 399710


In [10]:
# CELL 7 — Train, validate, save best checkpoint
EPOCHS = 30
LR = 1e-3
LAMBDA_MI = 0.1
LAMBDA_CLUSTER = 0.1
LAMBDA_CONSISTENCY = 0.1

gdev = {
    k: (v.to(DEVICE) if torch.is_tensor(v) else v)
    for k,v in graph.items()
}
y = gdev["labels"]
tr = gdev["train_idx"]; va = gdev["val_idx"]
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
best_val = float("inf")
history = []

for epoch in range(1, EPOCHS+1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    pred, aux = model(gdev, return_aux=True)
    supervised = F.mse_loss(pred[tr], y[tr])
    loss = (supervised + LAMBDA_MI*aux["mi"] +
            LAMBDA_CLUSTER*aux["cluster"] +
            LAMBDA_CONSISTENCY*aux["consistency"])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_pred = model(gdev)[va]
        val_rmse = torch.sqrt(F.mse_loss(val_pred, y[va])).item()
    history.append({"epoch":epoch, "loss":loss.item(), "supervised":supervised.item(),
                    "mi":aux["mi"].item(), "cluster":aux["cluster"].item(),
                    "consistency":aux["consistency"].item(), "val_rmse":val_rmse})
    print(f"Epoch {epoch:03d}/{EPOCHS} | loss={loss.item():.5f} "
          f"| val_RMSE={val_rmse:.5f}")
    if val_rmse < best_val:
        best_val = val_rmse
        torch.save({"model":model.state_dict(), "epoch":epoch, "val_rmse":best_val},
                   RESULT_DIR/"cate_hgn_best.pt")

pd.DataFrame(history).to_csv(RESULT_DIR/"training_history.csv", index=False)
print("Best validation RMSE:", best_val)


Epoch 001/30 | loss=49.99958 | val_RMSE=1.82399
Epoch 002/30 | loss=49.95326 | val_RMSE=1.81703
Epoch 003/30 | loss=49.90777 | val_RMSE=1.81015
Epoch 004/30 | loss=49.86282 | val_RMSE=1.80331
Epoch 005/30 | loss=49.81799 | val_RMSE=1.79640
Epoch 006/30 | loss=49.77265 | val_RMSE=1.78945
Epoch 007/30 | loss=49.72681 | val_RMSE=1.78236
Epoch 008/30 | loss=49.68010 | val_RMSE=1.77517
Epoch 009/30 | loss=49.63223 | val_RMSE=1.76782
Epoch 010/30 | loss=49.58329 | val_RMSE=1.76033
Epoch 011/30 | loss=49.53320 | val_RMSE=1.75255
Epoch 012/30 | loss=49.48113 | val_RMSE=1.74457
Epoch 013/30 | loss=49.42753 | val_RMSE=1.73635
Epoch 014/30 | loss=49.37203 | val_RMSE=1.72784
Epoch 015/30 | loss=49.31380 | val_RMSE=1.71905
Epoch 016/30 | loss=49.25311 | val_RMSE=1.70998
Epoch 017/30 | loss=49.18998 | val_RMSE=1.70068
Epoch 018/30 | loss=49.12445 | val_RMSE=1.69106
Epoch 019/30 | loss=49.05640 | val_RMSE=1.68107
Epoch 020/30 | loss=48.98466 | val_RMSE=1.67074
Epoch 021/30 | loss=48.90891 | val_RMSE=

In [11]:
# CELL 8 — Test evaluation and paper comparison
checkpoint = torch.load(RESULT_DIR/"cate_hgn_best.pt", map_location=DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

with torch.no_grad():
    all_pred = model(gdev).detach().cpu().numpy()
all_y = graph["labels"].cpu().numpy()

rows_out = []
for split_name, idx_tensor in [
    ("train", graph["train_idx"]), ("validation", graph["val_idx"]),
    ("test", graph["test_idx"])
]:
    idx = idx_tensor.cpu().numpy()
    if len(idx):
        rmse = math.sqrt(mean_squared_error(all_y[idx], all_pred[idx]))
        mae = mean_absolute_error(all_y[idx], all_pred[idx])
    else:
        rmse, mae = float("nan"), float("nan")
    rows_out.append({"split":split_name, "n":len(idx), "RMSE":rmse, "MAE":mae})
    print(f"{split_name:10s} n={len(idx):6d} RMSE={rmse:.5f} MAE={mae:.5f}")

metrics = pd.DataFrame(rows_out)
metrics.to_csv(RESULT_DIR/"metrics.csv", index=False)

# Reference values from CATE-HGN paper Table II. These are not our measurements.
paper_reference = pd.DataFrame([
    {"dataset":"DBLP-full", "paper_CATE_HGN_RMSE":3.4574},
    {"dataset":"DBLP-single", "paper_CATE_HGN_RMSE":4.8305},
    {"dataset":"DBLP-random", "paper_CATE_HGN_RMSE":3.4574},
])
print("\nPaper reference values (different dataset scale/settings):")
display(paper_reference)
print("\nOur measured results:")
display(metrics)
print("NOTE: subset result is not directly comparable to full/single/random paper settings.")


train      n=   341 RMSE=6.91258 MAE=1.71183
validation n=    42 RMSE=1.56323 MAE=1.10531
test       n=   117 RMSE=1.50855 MAE=0.74237

Paper reference values (different dataset scale/settings):


,dataset,paper_CATE_HGN_RMSE
0,DBLP-full,3.4574
1,DBLP-single,4.8305
2,DBLP-random,3.4574



Our measured results:


,split,n,RMSE,MAE
0,train,341,6.912579,1.711832
1,validation,42,1.563231,1.105306
2,test,117,1.508551,0.742372


NOTE: subset result is not directly comparable to full/single/random paper settings.


In [12]:
# CELL 9 — Ablation experiments (retrain each variant)
# These variants isolate the training losses. The graph encoder remains the same.
# Run this after the main experiment; it adds training time.

def train_ablation(name, use_mi=True, use_cluster=True, use_consistency=True, epochs=15):
    torch.manual_seed(SEED); np.random.seed(SEED)
    m = CATEHGNColab(n_papers, len(author_to_id), len(venue_to_id), len(terms),
                     dim=100, clusters=10, layers=2).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-5)
    best = float("inf")
    for ep in range(epochs):
        m.train(); opt.zero_grad(set_to_none=True)
        p, ax = m(gdev, return_aux=True)
        sup = F.mse_loss(p[tr], y[tr])
        loss = sup
        if use_mi: loss = loss + LAMBDA_MI*ax["mi"]
        if use_cluster: loss = loss + LAMBDA_CLUSTER*ax["cluster"]
        if use_consistency: loss = loss + LAMBDA_CONSISTENCY*ax["consistency"]
        loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step()
        m.eval()
        with torch.no_grad():
            vp = m(gdev)[va]
            score = torch.sqrt(F.mse_loss(vp,y[va])).item()
        best = min(best,score)
    return {"variant":name,"best_validation_RMSE":best}

# Uncomment to execute:
# ablation_results = [
#     train_ablation("Full CATE-HGN-style"),
#     train_ablation("Without MI", use_mi=False),
#     train_ablation("Without cluster loss", use_cluster=False),
#     train_ablation("Without consistency", use_consistency=False),
#     train_ablation("HGN-only", use_mi=False, use_cluster=False, use_consistency=False),
# ]
# pd.DataFrame(ablation_results).to_csv(RESULT_DIR/"ablation.csv", index=False)
# display(pd.DataFrame(ablation_results))


In [13]:
# CELL 10 — Download results from Colab
from google.colab import files
import shutil
bundle = Path("/content/cate_hgn_results.zip")
shutil.make_archive(str(bundle.with_suffix("")), "zip", RESULT_DIR)
files.download(str(bundle))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>